In [95]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from keras.layers import Dense, Activation, Dropout
import time #helper libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from keras.layers import Dense, Activation, Dropout
import time #helper libraries
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# combine stock data

### STOCK TIMEZONE: UTC (+00:00 is in the timestamp itself)
UTC (+00:00 and the Z both mean UTC).

In [96]:
df = pd.read_csv("data/stocks/ALPACA_hourly_AAPL_2018_2021.csv")
print(df.columns.tolist())
print(df.head(2))

['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']
  symbol                  timestamp    open    high    low   close  volume  \
0   AAPL  2018-01-02 09:00:00+00:00  169.10  169.20  169.1  169.18  8446.0   
1   AAPL  2018-01-02 10:00:00+00:00  169.25  169.35  169.0  169.35  5407.0   

   trade_count        vwap  
0         40.0  169.116356  
1         48.0  169.118009  


In [97]:
import pandas as pd
import os

files = {
    "AAPL": [
        "ALPACA_hourly_AAPL_2018_2021.csv",
        "ALPACA_hourly_AAPL_2022_2025.csv"
    ],
    "AMZN": [
        "ALPACA_hourly_AMZN_2018_2021.csv",
        "ALPACA_hourly_AMZN_2022_2025.csv"
    ],
    "TSLA": [
        "ALPACA_hourly_TSLA_2018_2021.csv",
        "ALPACA_hourly_TSLA_2022_2025.csv"
    ]
}

DATA_DIR = "data/stocks/"  # adjust path if needed

all_dfs = []

for ticker, file_list in files.items():
    for file in file_list:
        df = pd.read_csv(os.path.join(DATA_DIR, file))
        df = df[["timestamp", "close"]]   # adjust column name if needed
        df["ticker"] = ticker
        all_dfs.append(df)

# Stack all rows
combined = pd.concat(all_dfs, ignore_index=True)

# Pivot so each ticker is a column, rows are timestamps
combined["timestamp"] = pd.to_datetime(combined["timestamp"])
result = combined.pivot_table(index="timestamp", columns="ticker", values="close")
result.columns.name = None
result = result.sort_index()

print(result.shape)
print(result.head())

result.to_csv("stocks_combined.csv")

(31528, 3)
                             AAPL     AMZN    TSLA
timestamp                                         
2018-01-02 09:00:00+00:00  169.18  1170.00  311.80
2018-01-02 10:00:00+00:00  169.35  1169.70  311.50
2018-01-02 11:00:00+00:00  169.67      NaN  311.89
2018-01-02 12:00:00+00:00  170.05  1173.89  311.00
2018-01-02 13:00:00+00:00  170.12  1174.50  311.50


In [98]:
result = result.reset_index()
result = result[~result["timestamp"].dt.year.isin([2024, 2025])]
print(result.shape)
result.to_csv("stocks_combined_3companies_2018_2023.csv")

(23531, 4)


In [99]:
df_stocks = result

## merge news and textual processing

#### NEW TIMEZONE:  UTC timestamp in ISO 8601 (YYYY-MM-DDTHH:MM:SSZ). If original precision is only day-level, the top-level date is set to midnight UTC and trading date policy information is kept in extra_fields (see below).
UTC (+00:00 and the Z both mean UTC).

In [100]:
import pandas as pd
import glob

files = glob.glob("data/news/*.csv")  # change extension if needed

df_inds = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(df_inds.shape)

(147383, 5)


In [101]:
df_inds

,date,text,processed_Tesla,processed_Apple,processed_Amazon
0,2020-10-03 00:00:00+00:00,US President Donald Trump tests positive for C...,-4\n2\n1\n3\n5\n-2\n0\n2\n-1\n4,-5 -2 0 1 3 2 4 5 0 -1,-2\n3\n-1\n0\n4\n5\n2\n-3\n1\n-4
1,2020-10-08 00:00:00+00:00,"Guitarist Eddie Van Halen dies, aged 65\n\nObi...",-1\n5\n-2\n4\n3\n-1\n-1\n1\n5,-1\n2\n4\n3\n5\n6\n7\n8\n10\n5,3 4 -2 5 1 -3 2 4 3
2,2020-10-12 00:00:00+00:00,Second US Biden-Trump debate cancelled amid sa...,3 4 -1 2 5 0 1 -3 4 5,-2 -3 1 4 0 -5 2 5 1 -1,3\n2\n5\n4\n1\n-1\n3\n-1\n0\n5
3,2020-10-19 00:00:00+00:00,"Wikinews interviews Don Blankenship, U.S. Cons...",-3\n4\n1\n5\n2\n0\n3\n5\n0\n2,-2\n0\n3\n4\n1\n5\n2\n-1\n0\n5,-3\n0\n1\n2\n5\n4\n3\n0\n-1\n1
4,2020-10-22 00:00:00+00:00,"Wikinews interviews Phil Collins, U.S. Prohibi...",-1\n3\n2\n4\n5\n0\n3\n-1\n6\n-2,4 2 -3 0 5 1 2 5 3 1,3 0 1 -4 2 5 4 -1 5
...,...,...,...,...,...
147378,2022-10-05 15:27:02+00:00,"Amazon adds free movie trailers, lifestyle con...",2 0 -3 1 4 -5 2 3 1 1,-3\n0\n2\n0\n1\n0\n-1\n4\n0\n5,-1\n0\n3\n2\n1\n0\n1\n4\n2\n5
147379,2022-10-05 15:09:17+00:00,Stadia launch title 'Gylt' is heading to other...,-2 -1 0 4 5 1 2 2 1 1,-5 -2 1 4 3 0 5 1 0 2,-4\n1\n0\n5\n2\n0\n0\n3\n4\n2
147380,2022-10-05 14:57:07+00:00,"Amazon kills its interactive, video-calling de...",-2\n1\n4\n3\n5\n0\n0\n2\n5\n1,-1\n3\n4\n0\n2\n5\n0\n-1\n0\n0\n0,-2\n0\n3\n4\n2\n1\n-1\n5\n2\n1
147381,2022-10-05 14:18:38+00:00,Amazon's Blink security cameras and bundles ar...,3 -5 4 1 -2 2 5 -1 3 -5,-3 -1 2 0 4 1 2 5 3 4,2 -1 3 4 5 -2 6 -1 8 -1


In [102]:
df_inds["date"] = pd.to_datetime(df_inds["date"])
df_inds["date"].dt.to_period("M").unique()

/tmp/ipykernel_3706123/2166345840.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_inds["date"].dt.to_period("M").unique()


<PeriodArray>
['2020-10', '2018-03', '2021-08', '2022-11', '2020-04', '2023-11', '2022-03',
 '2023-03', '2022-12', '2023-01', '2020-08', '2023-05', '2019-12', '2018-06',
 '2019-02', '2019-05', '2019-07', '2021-03', '2021-12', '2023-09', '2022-04',
 '2019-10', '2019-08', '2022-05', '2020-12', '2022-07', '2021-09', '2018-01',
 '2018-08', '2020-09', '2018-10', '2023-04', '2021-06', '2020-03', '2022-02',
 '2019-04', '2023-07', '2021-04', '2018-07', '2022-09', '2022-08', '2022-06',
 '2021-05', '2020-02', '2020-01', '2020-11', '2018-11', '2023-10', '2019-01',
 '2023-02', '2022-01', '2018-05', '2021-07', '2018-02', '2021-01', '2021-11',
 '2018-04', '2020-05', '2020-07', '2021-10', '2019-11', '2023-12', '2019-09',
 '2018-09', '2023-08', '2018-12', '2019-06', '2020-06', '2021-02', '2019-03',
 '2023-06', '2022-10']
Length: 72, dtype: period[M]

In [103]:
df_inds

,date,text,processed_Tesla,processed_Apple,processed_Amazon
0,2020-10-03 00:00:00+00:00,US President Donald Trump tests positive for C...,-4\n2\n1\n3\n5\n-2\n0\n2\n-1\n4,-5 -2 0 1 3 2 4 5 0 -1,-2\n3\n-1\n0\n4\n5\n2\n-3\n1\n-4
1,2020-10-08 00:00:00+00:00,"Guitarist Eddie Van Halen dies, aged 65\n\nObi...",-1\n5\n-2\n4\n3\n-1\n-1\n1\n5,-1\n2\n4\n3\n5\n6\n7\n8\n10\n5,3 4 -2 5 1 -3 2 4 3
2,2020-10-12 00:00:00+00:00,Second US Biden-Trump debate cancelled amid sa...,3 4 -1 2 5 0 1 -3 4 5,-2 -3 1 4 0 -5 2 5 1 -1,3\n2\n5\n4\n1\n-1\n3\n-1\n0\n5
3,2020-10-19 00:00:00+00:00,"Wikinews interviews Don Blankenship, U.S. Cons...",-3\n4\n1\n5\n2\n0\n3\n5\n0\n2,-2\n0\n3\n4\n1\n5\n2\n-1\n0\n5,-3\n0\n1\n2\n5\n4\n3\n0\n-1\n1
4,2020-10-22 00:00:00+00:00,"Wikinews interviews Phil Collins, U.S. Prohibi...",-1\n3\n2\n4\n5\n0\n3\n-1\n6\n-2,4 2 -3 0 5 1 2 5 3 1,3 0 1 -4 2 5 4 -1 5
...,...,...,...,...,...
147378,2022-10-05 15:27:02+00:00,"Amazon adds free movie trailers, lifestyle con...",2 0 -3 1 4 -5 2 3 1 1,-3\n0\n2\n0\n1\n0\n-1\n4\n0\n5,-1\n0\n3\n2\n1\n0\n1\n4\n2\n5
147379,2022-10-05 15:09:17+00:00,Stadia launch title 'Gylt' is heading to other...,-2 -1 0 4 5 1 2 2 1 1,-5 -2 1 4 3 0 5 1 0 2,-4\n1\n0\n5\n2\n0\n0\n3\n4\n2
147380,2022-10-05 14:57:07+00:00,"Amazon kills its interactive, video-calling de...",-2\n1\n4\n3\n5\n0\n0\n2\n5\n1,-1\n3\n4\n0\n2\n5\n0\n-1\n0\n0\n0,-2\n0\n3\n4\n2\n1\n-1\n5\n2\n1
147381,2022-10-05 14:18:38+00:00,Amazon's Blink security cameras and bundles ar...,3 -5 4 1 -2 2 5 -1 3 -5,-3 -1 2 0 4 1 2 5 3 4,2 -1 3 4 5 -2 6 -1 8 -1


# Cleaning up the df files an merging

In [104]:
#df_inds= pd.read_csv("/kaggle/input/cdl-fall25-textual/Brianferrell_analyzed_2023.csv")

In [105]:
#df_stocks = pd.read_csv("/kaggle/input/cdl-fall25-textual/huggingface_mdnh_hourly-stock-data-2023.csv")

In [106]:
#df_inds["scores"] = df_inds["processed"].str.findall(r"-?\d+")
for col in ["processed_Tesla", "processed_Apple", "processed_Amazon"]:
    df_inds[f"scores_{col}"] = df_inds[col].str.findall(r"-?\d+")

In [107]:
df_inds["processed_Tesla"]

0         -4\n2\n1\n3\n5\n-2\n0\n2\n-1\n4
1           -1\n5\n-2\n4\n3\n-1\n-1\n1\n5
2                   3 4 -1 2 5 0 1 -3 4 5
3           -3\n4\n1\n5\n2\n0\n3\n5\n0\n2
4         -1\n3\n2\n4\n5\n0\n3\n-1\n6\n-2
                       ...               
147378              2 0 -3 1 4 -5 2 3 1 1
147379              -2 -1 0 4 5 1 2 2 1 1
147380      -2\n1\n4\n3\n5\n0\n0\n2\n5\n1
147381            3 -5 4 1 -2 2 5 -1 3 -5
147382    -3\n2\n0\n0\n1\n-4\n0\n-2\n0\n5
Name: processed_Tesla, Length: 147383, dtype: object

In [108]:
df_inds["scores_processed_Tesla"]

0          [-4, 2, 1, 3, 5, -2, 0, 2, -1, 4]
1            [-1, 5, -2, 4, 3, -1, -1, 1, 5]
2           [3, 4, -1, 2, 5, 0, 1, -3, 4, 5]
3            [-3, 4, 1, 5, 2, 0, 3, 5, 0, 2]
4          [-1, 3, 2, 4, 5, 0, 3, -1, 6, -2]
                         ...                
147378      [2, 0, -3, 1, 4, -5, 2, 3, 1, 1]
147379      [-2, -1, 0, 4, 5, 1, 2, 2, 1, 1]
147380       [-2, 1, 4, 3, 5, 0, 0, 2, 5, 1]
147381    [3, -5, 4, 1, -2, 2, 5, -1, 3, -5]
147382     [-3, 2, 0, 0, 1, -4, 0, -2, 0, 5]
Name: scores_processed_Tesla, Length: 147383, dtype: object

In [109]:
df_inds.columns
df_inds.head()


,date,text,processed_Tesla,processed_Apple,processed_Amazon,scores_processed_Tesla,scores_processed_Apple,scores_processed_Amazon
0,2020-10-03 00:00:00+00:00,US President Donald Trump tests positive for C...,-4\n2\n1\n3\n5\n-2\n0\n2\n-1\n4,-5 -2 0 1 3 2 4 5 0 -1,-2\n3\n-1\n0\n4\n5\n2\n-3\n1\n-4,"[-4, 2, 1, 3, 5, -2, 0, 2, -1, 4]","[-5, -2, 0, 1, 3, 2, 4, 5, 0, -1]","[-2, 3, -1, 0, 4, 5, 2, -3, 1, -4]"
1,2020-10-08 00:00:00+00:00,"Guitarist Eddie Van Halen dies, aged 65\n\nObi...",-1\n5\n-2\n4\n3\n-1\n-1\n1\n5,-1\n2\n4\n3\n5\n6\n7\n8\n10\n5,3 4 -2 5 1 -3 2 4 3,"[-1, 5, -2, 4, 3, -1, -1, 1, 5]","[-1, 2, 4, 3, 5, 6, 7, 8, 10, 5]","[3, 4, -2, 5, 1, -3, 2, 4, 3]"
2,2020-10-12 00:00:00+00:00,Second US Biden-Trump debate cancelled amid sa...,3 4 -1 2 5 0 1 -3 4 5,-2 -3 1 4 0 -5 2 5 1 -1,3\n2\n5\n4\n1\n-1\n3\n-1\n0\n5,"[3, 4, -1, 2, 5, 0, 1, -3, 4, 5]","[-2, -3, 1, 4, 0, -5, 2, 5, 1, -1]","[3, 2, 5, 4, 1, -1, 3, -1, 0, 5]"
3,2020-10-19 00:00:00+00:00,"Wikinews interviews Don Blankenship, U.S. Cons...",-3\n4\n1\n5\n2\n0\n3\n5\n0\n2,-2\n0\n3\n4\n1\n5\n2\n-1\n0\n5,-3\n0\n1\n2\n5\n4\n3\n0\n-1\n1,"[-3, 4, 1, 5, 2, 0, 3, 5, 0, 2]","[-2, 0, 3, 4, 1, 5, 2, -1, 0, 5]","[-3, 0, 1, 2, 5, 4, 3, 0, -1, 1]"
4,2020-10-22 00:00:00+00:00,"Wikinews interviews Phil Collins, U.S. Prohibi...",-1\n3\n2\n4\n5\n0\n3\n-1\n6\n-2,4 2 -3 0 5 1 2 5 3 1,3 0 1 -4 2 5 4 -1 5,"[-1, 3, 2, 4, 5, 0, 3, -1, 6, -2]","[4, 2, -3, 0, 5, 1, 2, 5, 3, 1]","[3, 0, 1, -4, 2, 5, 4, -1, 5]"


In [110]:
df_inds = df_inds.drop(columns=["processed_Tesla", "processed_Apple", "processed_Amazon"])

In [111]:
df_inds.head()


,date,text,scores_processed_Tesla,scores_processed_Apple,scores_processed_Amazon
0,2020-10-03 00:00:00+00:00,US President Donald Trump tests positive for C...,"[-4, 2, 1, 3, 5, -2, 0, 2, -1, 4]","[-5, -2, 0, 1, 3, 2, 4, 5, 0, -1]","[-2, 3, -1, 0, 4, 5, 2, -3, 1, -4]"
1,2020-10-08 00:00:00+00:00,"Guitarist Eddie Van Halen dies, aged 65\n\nObi...","[-1, 5, -2, 4, 3, -1, -1, 1, 5]","[-1, 2, 4, 3, 5, 6, 7, 8, 10, 5]","[3, 4, -2, 5, 1, -3, 2, 4, 3]"
2,2020-10-12 00:00:00+00:00,Second US Biden-Trump debate cancelled amid sa...,"[3, 4, -1, 2, 5, 0, 1, -3, 4, 5]","[-2, -3, 1, 4, 0, -5, 2, 5, 1, -1]","[3, 2, 5, 4, 1, -1, 3, -1, 0, 5]"
3,2020-10-19 00:00:00+00:00,"Wikinews interviews Don Blankenship, U.S. Cons...","[-3, 4, 1, 5, 2, 0, 3, 5, 0, 2]","[-2, 0, 3, 4, 1, 5, 2, -1, 0, 5]","[-3, 0, 1, 2, 5, 4, 3, 0, -1, 1]"
4,2020-10-22 00:00:00+00:00,"Wikinews interviews Phil Collins, U.S. Prohibi...","[-1, 3, 2, 4, 5, 0, 3, -1, 6, -2]","[4, 2, -3, 0, 5, 1, 2, 5, 3, 1]","[3, 0, 1, -4, 2, 5, 4, -1, 5]"


In [112]:
df_inds.to_csv("Brianferrell_processednews_llama32_3companies_2018_2023.csv")

In [113]:
df_stocks.columns
df_stocks.head()
#df_stocks.size

,timestamp,AAPL,AMZN,TSLA
0,2018-01-02 09:00:00+00:00,169.18,1170.00,311.80
1,2018-01-02 10:00:00+00:00,169.35,1169.70,311.50
2,2018-01-02 11:00:00+00:00,169.67,NaN,311.89
3,2018-01-02 12:00:00+00:00,170.05,1173.89,311.00
4,2018-01-02 13:00:00+00:00,170.12,1174.50,311.50


In [114]:
df_stocks.drop(columns=["index"], inplace=True)

KeyError: "['index'] not found in axis"

In [ ]:
df_stocks.head()


In [ ]:
#df_stocks = df_stocks[df_stocks["symbol"]=="TSLA"]

In [ ]:
#df_inds.drop(columns=["text", "Impact Scores", "processed"], inplace=True)

In [ ]:
#df_inds.head()

In [ ]:
#df_stocks.size

## merge

In [ ]:
df_stocks["timestamp"] = pd.to_datetime(df_stocks["timestamp"])


In [ ]:
df_stocks["timestamp"]

In [ ]:
df_inds["date"] = pd.to_datetime(df_inds["date"])

In [ ]:
df_stocks["day"] = df_stocks["timestamp"].dt.date
df_stocks["hour"] = df_stocks["timestamp"].dt.hour

df_inds["day"] = df_inds["date"].dt.date
df_inds["hour"] = df_inds["date"].dt.hour


In [ ]:
df_inds["hour"]

In [ ]:
df_inds["seq"] = df_inds.groupby(["day", "hour"]).cumcount()

In [ ]:
df_inds["seq"]

In [ ]:
df_inds_lists = (
    df_inds
    .sort_values(["day", "hour"])
    .groupby(["day", "hour"], sort=False)
    .agg(
        scores_Tesla=("scores_processed_Tesla", list),
        scores_Apple=("scores_processed_Apple", list),
        scores_Amazon=("scores_processed_Amazon", list),
    )
    .reset_index()
)

In [115]:
df_inds_lists.to_csv("Brianferrell_processednews_llama32_3companies_2018_2023_by_day_hour.csv", index=False)

In [116]:
# 3) expand those lists into separate columns processed_0, processed_1, ...
'''
indicators_expanded = pd.DataFrame(
    df_inds_lists["scores_list"].tolist()
).rename(columns=lambda i: f"scores_{i}")
'''

'\nindicators_expanded = pd.DataFrame(\n    df_inds_lists["scores_list"].tolist()\n).rename(columns=lambda i: f"scores_{i}")\n'

In [117]:
#indicators_expanded

In [118]:
'''
df_inds_wide = pd.concat([df_inds_lists[["day", "hour"]].reset_index(drop=True),
                           indicators_expanded.reset_index(drop=True)],
                          axis=1)
                          
'''

'\ndf_inds_wide = pd.concat([df_inds_lists[["day", "hour"]].reset_index(drop=True),\n                           indicators_expanded.reset_index(drop=True)],\n                          axis=1)\n                          \n'

In [119]:
#df_inds_wide

In [120]:
df_stocks["day"] = df_stocks["timestamp"].dt.date
df_stocks["hour"] = df_stocks["timestamp"].dt.hour



In [121]:
df_stocks

,timestamp,AAPL,AMZN,TSLA,day,hour
0,2018-01-02 09:00:00+00:00,169.1800,1170.0000,311.80,2018-01-02,9
1,2018-01-02 10:00:00+00:00,169.3500,1169.7000,311.50,2018-01-02,10
2,2018-01-02 11:00:00+00:00,169.6700,NaN,311.89,2018-01-02,11
3,2018-01-02 12:00:00+00:00,170.0500,1173.8900,311.00,2018-01-02,12
4,2018-01-02 13:00:00+00:00,170.1200,1174.5000,311.50,2018-01-02,13
...,...,...,...,...,...,...
23526,2023-12-29 20:00:00+00:00,192.5200,151.9500,248.49,2023-12-29,20
23527,2023-12-29 21:00:00+00:00,192.3800,151.7600,248.40,2023-12-29,21
23528,2023-12-29 22:00:00+00:00,192.2600,151.7485,248.41,2023-12-29,22
23529,2023-12-29 23:00:00+00:00,192.1894,151.7700,248.38,2023-12-29,23


In [122]:
#df = df_stocks.merge(df_inds_wide, on=["day", "hour"], how="left")
df = df_stocks.merge(df_inds_lists, on=["day", "hour"], how="outer")


In [123]:
df.columns

Index(['timestamp', 'AAPL', 'AMZN', 'TSLA', 'day', 'hour', 'scores_Tesla',
       'scores_Apple', 'scores_Amazon'],
      dtype='object')

In [124]:
df.to_csv("processed_news_scores_with_stocks_organized_by_hour_day_combination.csv", index=False)

In [127]:
df

,timestamp,AAPL,AMZN,TSLA,day,hour,scores_Tesla,scores_Apple,scores_Amazon
0,2018-01-02 09:00:00+00:00,169.18,1170.00,311.80,2018-01-02,9,NaN,NaN,NaN
1,2018-01-02 10:00:00+00:00,169.35,1169.70,311.50,2018-01-02,10,NaN,NaN,NaN
2,2018-01-02 11:00:00+00:00,169.67,NaN,311.89,2018-01-02,11,NaN,NaN,NaN
3,2018-01-02 12:00:00+00:00,170.05,1173.89,311.00,2018-01-02,12,NaN,NaN,NaN
4,2018-01-02 13:00:00+00:00,170.12,1174.50,311.50,2018-01-02,13,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
31787,NaT,NaN,NaN,NaN,2023-12-30,14,"[[3, -5, 0, 1, -4, 0, 4, -3, 0, 2], [-2, 0, 5,...","[[-3, -1, 0, 2, 1, 2, 0, 0, 5, 3], [3, 0, -1, ...","[[-1, 3, 0, 5, 2, 0, 0, -1, 3, 4], [-3, -1, 0,..."
31788,NaT,NaN,NaN,NaN,2023-12-30,15,"[[-3, 1, 5, 2, 4, 0, 1, 3, 0, 0], [-2, 5, 3, 1...","[[2, 4, -1, 3, 5, 0, 2, 1, 4, 5], [3, 0, -1, 2...","[[-2, 0, 3, 1, 5, 4, -2], [3, 0, -2, 5, 0, 2, ..."
31789,NaT,NaN,NaN,NaN,2023-12-30,16,"[[-3, -1, 0, 2, 3, 4, 0, 2, 2, 0]]","[[-3, 2, 0, 4, 1, -2, -1, 0, -5, 5]]","[[-4, 0, 3, 2, 1, 0, 1, 2, 0, 1]]"
31790,NaT,NaN,NaN,NaN,2023-12-30,17,"[[-3, 2, 5, 1, 0, -2, 4, -1, 2, 1], [-2, 3, 4,...","[[3, 1, 2, -5, 0, -4, 1, 0, 5, -2], [3, -2, 5,...","[[-2, 3, 0, 1, 4, 5, 2, -1, 0, 5], [3, 2, -5, ..."


In [125]:
df[5:20]

,timestamp,AAPL,AMZN,TSLA,day,hour,scores_Tesla,scores_Apple,scores_Amazon
5,2018-01-02 14:00:00+00:00,170.9800,1181.4300,317.330,2018-01-02,14,NaN,NaN,NaN
6,2018-01-02 15:00:00+00:00,171.5050,1182.8200,318.050,2018-01-02,15,NaN,NaN,NaN
7,2018-01-02 16:00:00+00:00,171.2100,1187.3086,320.515,2018-01-02,16,NaN,NaN,NaN
8,2018-01-02 17:00:00+00:00,171.2800,1187.1200,319.390,2018-01-02,17,NaN,NaN,NaN
9,2018-01-02 18:00:00+00:00,171.6886,1187.2800,319.720,2018-01-02,18,NaN,NaN,NaN
10,2018-01-02 19:00:00+00:00,171.9800,1188.6750,320.440,2018-01-02,19,NaN,NaN,NaN
11,2018-01-02 20:00:00+00:00,172.2600,1189.0000,320.290,2018-01-02,20,NaN,NaN,NaN
12,2018-01-02 21:00:00+00:00,172.2000,1186.0000,320.530,2018-01-02,21,NaN,NaN,NaN
13,2018-01-02 22:00:00+00:00,172.1100,1186.4200,320.750,2018-01-02,22,NaN,NaN,NaN
14,2018-01-02 23:00:00+00:00,172.1500,1186.0500,320.510,2018-01-02,23,NaN,NaN,NaN


In [126]:
df.dropna().shape[0]

9566

In [142]:
df_apple = df[["timestamp", "AAPL", "day", "hour", "scores_Apple"]].copy()
df_amazon = df[["timestamp", "AMZN", "day", "hour", "scores_Amazon"]].copy()
df_tesla = df[["timestamp", "TSLA", "day", "hour", "scores_Tesla"]].copy()

print(df_apple.shape, df_amazon.shape, df_tesla.shape)

(31792, 5) (31792, 5) (31792, 5)


 ###  Accumulating all news when the market was shut down instead of throwing it out and combining it so when the market opens, all the news of the last 9 hours have an impact.
 #### for each company separated

In [128]:
'''
result = []
accumulated = {"scores_Tesla": [], "scores_Apple": [], "scores_Amazon": []}

for _, row in df.iterrows():
    if pd.isna(row["AAPL"]) and pd.isna(row["AMZN"]) and pd.isna(row["TSLA"]):
        for col in ["scores_Tesla", "scores_Apple", "scores_Amazon"]:
            if isinstance(row[col], list):
                accumulated[col].extend(row[col])
    else:
        row = row.copy()
        for col in ["scores_Tesla", "scores_Apple", "scores_Amazon"]:
            row[col] = accumulated[col] + (row[col] if isinstance(row[col], list) else [])
        accumulated = {"scores_Tesla": [], "scores_Apple": [], "scores_Amazon": []}
        result.append(row)

df_clean = pd.DataFrame(result).reset_index(drop=True)
print(df_clean.shape)
#(23531, 9)

'''

(23531, 9)


In [129]:
#df_clean

,timestamp,AAPL,AMZN,TSLA,day,hour,scores_Tesla,scores_Apple,scores_Amazon
0,2018-01-02 09:00:00+00:00,169.1800,1170.0000,311.80,2018-01-02,9,[],[],[]
1,2018-01-02 10:00:00+00:00,169.3500,1169.7000,311.50,2018-01-02,10,[],[],[]
2,2018-01-02 11:00:00+00:00,169.6700,NaN,311.89,2018-01-02,11,[],[],[]
3,2018-01-02 12:00:00+00:00,170.0500,1173.8900,311.00,2018-01-02,12,[],[],[]
4,2018-01-02 13:00:00+00:00,170.1200,1174.5000,311.50,2018-01-02,13,[],[],[]
...,...,...,...,...,...,...,...,...,...
23526,2023-12-29 20:00:00+00:00,192.5200,151.9500,248.49,2023-12-29,20,[],[],[]
23527,2023-12-29 21:00:00+00:00,192.3800,151.7600,248.40,2023-12-29,21,[],[],[]
23528,2023-12-29 22:00:00+00:00,192.2600,151.7485,248.41,2023-12-29,22,[],[],[]
23529,2023-12-29 23:00:00+00:00,192.1894,151.7700,248.38,2023-12-29,23,[],[],[]


In [143]:
def accumulate_scores(df, price_col, score_col):
    result = []
    accumulated = []
    for _, row in df.iterrows():
        if pd.isna(row[price_col]):
            if isinstance(row[score_col], list):
                accumulated.extend(row[score_col])
        else:
            row = row.copy()
            row[score_col] = accumulated + (row[score_col] if isinstance(row[score_col], list) else [])
            accumulated = []
            result.append(row)
    return pd.DataFrame(result).reset_index(drop=True)

df_apple_clean  = accumulate_scores(df_apple,  "AAPL", "scores_Apple")
df_amazon_clean = accumulate_scores(df_amazon, "AMZN", "scores_Amazon")
df_tesla_clean  = accumulate_scores(df_tesla,  "TSLA", "scores_Tesla")

print(df_apple_clean.shape, df_amazon_clean.shape, df_tesla_clean.shape)

(23500, 5) (22383, 5) (23412, 5)


In [144]:
df_apple_clean[["AAPL"]].isna().sum()

AAPL    0
dtype: int64

In [146]:
df_tesla_clean[["TSLA"]].isna().sum()

TSLA    0
dtype: int64

In [147]:
df_amazon_clean[["AMZN"]].isna().sum()

AMZN    0
dtype: int64

## Handling nan

so I want to handle nan in different ways. 1- if consecutive rows of 3 or more,  and if other columns have nan as well, drop rows. If only one column has nan, and it's in between two non-nan values, use linear interpolation. 

#### not needed anymore when each company was separated. 

In [138]:
'''
cols = ["AAPL","AMZN","TSLA"]

mask = df_clean[cols].isna()

# compute run lengths per column
run_lengths = mask.apply(lambda s: s.groupby((~s).cumsum()).transform('sum'))

# rows to drop: ≥3 consecutive NaNs AND more than one column NaN
drop_rows = (run_lengths >= 3) & (mask.sum(axis=1) > 1)
df_clean = df_clean[~drop_rows.any(axis=1)]

# interpolate only small gaps (single NaNs between values)
df_clean[cols] = df_clean[cols].interpolate(method='linear', limit=1, limit_direction='both')
'''

In [139]:
#df_clean[["AAPL","AMZN","TSLA"]].isna().any()

AAPL    False
AMZN     True
TSLA     True
dtype: bool

In [140]:
#df_clean[["AAPL","AMZN","TSLA"]].isna().sum()

AAPL      0
AMZN    128
TSLA      2
dtype: int64

In [148]:
#df_clean[df_clean[["AAPL","AMZN","TSLA"]].isna().any(axis=1)]

In [79]:
#df.to_csv("2023_tesla_stocks_news_mixed.csv")
#df_clean.to_csv("processed_news_scores_with_stocks_organized_by_hour_day_combination_no_NAN.csv", index=False)

In [152]:
df_apple_clean["scores_Apple"]

0        []
1        []
2        []
3        []
4        []
         ..
23495    []
23496    []
23497    []
23498    []
23499    []
Name: scores_Apple, Length: 23500, dtype: object

In [154]:
#df_amazon_clean

In [155]:
df_tesla_clean.to_csv("Tesla_processed_news_scores_with_stocks_organized_by_hour_day_combination_no_NAN.csv", index=False)
df_apple_clean.to_csv("Apple_processed_news_scores_with_stocks_organized_by_hour_day_combination_no_NAN.csv", index=False)
df_amazon_clean.to_csv("Amazon_processed_news_scores_with_stocks_organized_by_hour_day_combination_no_NAN.csv", index=False)

### Trying aggregate scores combinations

In [80]:
'''
score_cols = [f"scores_{i}" for i in range(23)]
def normalize_list(lst):
    lst = np.array(lst, dtype=float)
    max_val = lst.max()
    min_val = lst.min()
    def norm(x):
        if x > 0:
            return x / max_val if max_val != 0 else 0
        elif x < 0:
            return x / abs(min_val) if min_val != 0 else 0
        else:
            return 0
    return np.array([norm(x) for x in lst])

def combine_and_normalize(row):
    lists = [np.array(row[col], dtype=float) for col in score_cols if isinstance(row[col], list)]
    if not lists:
        return np.zeros(10)
    summed = np.sum(lists, axis=0)
    return normalize_list(summed)
'''

'\nscore_cols = [f"scores_{i}" for i in range(23)]\ndef normalize_list(lst):\n    lst = np.array(lst, dtype=float)\n    max_val = lst.max()\n    min_val = lst.min()\n    def norm(x):\n        if x > 0:\n            return x / max_val if max_val != 0 else 0\n        elif x < 0:\n            return x / abs(min_val) if min_val != 0 else 0\n        else:\n            return 0\n    return np.array([norm(x) for x in lst])\n\ndef combine_and_normalize(row):\n    lists = [np.array(row[col], dtype=float) for col in score_cols if isinstance(row[col], list)]\n    if not lists:\n        return np.zeros(10)\n    summed = np.sum(lists, axis=0)\n    return normalize_list(summed)\n'

In [81]:
'''
df["normalized_list"] = df.apply(combine_and_normalize, axis=1)
df["avg_normalized"] = df["normalized_list"].apply(np.mean)
df["max_normalized"] = df["normalized_list"].apply(np.max)
df["min_normalized"] = df["normalized_list"].apply(np.min)
'''

'\ndf["normalized_list"] = df.apply(combine_and_normalize, axis=1)\ndf["avg_normalized"] = df["normalized_list"].apply(np.mean)\ndf["max_normalized"] = df["normalized_list"].apply(np.max)\ndf["min_normalized"] = df["normalized_list"].apply(np.min)\n'

In [82]:
'''
import numpy as np

def normalize_list(lst):
    lst = np.array(lst, dtype=float)
    max_val = lst.max()
    min_val = lst.min()
    def norm(x):
        if x > 0:
            return x / max_val if max_val != 0 else 0
        elif x < 0:
            return x / abs(min_val) if min_val != 0 else 0
        else:
            return 0
    return np.array([norm(x) for x in lst])

for stock, col in zip(["Tesla", "Apple", "Amazon"], ["scores_Tesla", "scores_Apple", "scores_Amazon"]):
    def process(row, col=col):
        lst = row[col]
        if not isinstance(lst, list) or len(lst) == 0:
            return pd.Series({f"avg_norm_{col}": np.nan, f"max_norm_{col}": np.nan, f"min_norm_{col}": np.nan})
        normalized = normalize_list([float(x) for x in lst])
        return pd.Series({
            f"avg_norm_{col}": np.mean(normalized),
            f"max_norm_{col}": np.max(normalized),
            f"min_norm_{col}": np.min(normalized),
        })

    df_clean[[f"avg_norm_{col}", f"max_norm_{col}", f"min_norm_{col}"]] = df_clean.apply(process, axis=1)

print(df_clean[["avg_norm_scores_Tesla", "max_norm_scores_Tesla", "min_norm_scores_Tesla",
                "avg_norm_scores_Apple", "max_norm_scores_Apple", "min_norm_scores_Apple",
                "avg_norm_scores_Amazon", "max_norm_scores_Amazon", "min_norm_scores_Amazon"]].head())
'''

'\nimport numpy as np\n\ndef normalize_list(lst):\n    lst = np.array(lst, dtype=float)\n    max_val = lst.max()\n    min_val = lst.min()\n    def norm(x):\n        if x > 0:\n            return x / max_val if max_val != 0 else 0\n        elif x < 0:\n            return x / abs(min_val) if min_val != 0 else 0\n        else:\n            return 0\n    return np.array([norm(x) for x in lst])\n\nfor stock, col in zip(["Tesla", "Apple", "Amazon"], ["scores_Tesla", "scores_Apple", "scores_Amazon"]):\n    def process(row, col=col):\n        lst = row[col]\n        if not isinstance(lst, list) or len(lst) == 0:\n            return pd.Series({f"avg_norm_{col}": np.nan, f"max_norm_{col}": np.nan, f"min_norm_{col}": np.nan})\n        normalized = normalize_list([float(x) for x in lst])\n        return pd.Series({\n            f"avg_norm_{col}": np.mean(normalized),\n            f"max_norm_{col}": np.max(normalized),\n            f"min_norm_{col}": np.min(normalized),\n        })\n\n    df_clean[

#### scalars
avg_norm_avg_Tesla = avg of normalized avg-per-innerlist

avg_norm_max_Tesla = avg of normalized max-per-innerlist

avg_norm_min_Tesla = avg of normalized min-per-innerlist

In [83]:
'''
import itertools

for stock, col in zip(["Tesla", "Apple", "Amazon"], ["scores_Tesla", "scores_Apple", "scores_Amazon"]):
    def process(row, col=col):
        lst = row[col]
        if not isinstance(lst, list) or len(lst) == 0:
            return pd.Series({f"avg_norm_{col}": np.nan, f"max_norm_{col}": np.nan, f"min_norm_{col}": np.nan})
        # Flatten nested lists
        flat = list(itertools.chain.from_iterable(x if isinstance(x, list) else [x] for x in lst))
        if len(flat) == 0:
            return pd.Series({f"avg_norm_{col}": np.nan, f"max_norm_{col}": np.nan, f"min_norm_{col}": np.nan})
        normalized = normalize_list([float(x) for x in flat])
        return pd.Series({
            f"avg_norm_{col}": np.mean(normalized),
            f"max_norm_{col}": np.max(normalized),
            f"min_norm_{col}": np.min(normalized),
        })

    df_clean[[f"avg_norm_{col}", f"max_norm_{col}", f"min_norm_{col}"]] = df_clean.apply(process, axis=1)
'''



   avg_norm_avg_scores_Tesla  avg_norm_max_scores_Tesla  \
0                        NaN                        NaN   
1                        NaN                        NaN   
2                        NaN                        NaN   
3                        NaN                        NaN   
4                        NaN                        NaN   

   avg_norm_min_scores_Tesla  avg_norm_avg_scores_Apple  \
0                        NaN                        NaN   
1                        NaN                        NaN   
2                        NaN                        NaN   
3                        NaN                        NaN   
4                        NaN                        NaN   

   avg_norm_max_scores_Apple  avg_norm_min_scores_Apple  \
0                        NaN                        NaN   
1                        NaN                        NaN   
2                        NaN                        NaN   
3                        NaN                        Na

In [156]:
'''
import numpy as np
import itertools

def normalize_list(lst):
    lst = np.array(lst, dtype=float)
    if len(lst) == 0:
        return lst
    max_val = lst.max()
    min_val = lst.min()
    def norm(x):
        if x > 0:
            return x / max_val if max_val != 0 else 0
        elif x < 0:
            return x / abs(min_val) if min_val != 0 else 0
        else:
            return 0
    return np.array([norm(x) for x in lst])

for stock, col in zip(["Tesla", "Apple", "Amazon"], ["scores_Tesla", "scores_Apple", "scores_Amazon"]):
    def process(row, col=col):
        lst = row[col]
        if not isinstance(lst, list) or len(lst) == 0:
            return pd.Series({
                f"avg_norm_avg_{col}": np.nan,
                f"avg_norm_max_{col}": np.nan,
                f"avg_norm_min_{col}": np.nan,
            })

        # For each inner list, compute avg, max, min
        avgs, maxs, mins = [], [], []
        for inner in lst:
            if isinstance(inner, list) and len(inner) > 0:
                arr = np.array(inner, dtype=float)
                avgs.append(np.mean(arr))
                maxs.append(np.max(arr))
                mins.append(np.min(arr))

        if len(avgs) == 0:
            return pd.Series({
                f"avg_norm_avg_{col}": np.nan,
                f"avg_norm_max_{col}": np.nan,
                f"avg_norm_min_{col}": np.nan,
            })

        return pd.Series({
            f"avg_norm_avg_{col}": np.mean(normalize_list(avgs)),
            f"avg_norm_max_{col}": np.mean(normalize_list(maxs)),
            f"avg_norm_min_{col}": np.mean(normalize_list(mins)),
        })

    df_clean[[f"avg_norm_avg_{col}", f"avg_norm_max_{col}", f"avg_norm_min_{col}"]] = df_clean.apply(process, axis=1)

print(df_clean[[
    "avg_norm_avg_scores_Tesla", "avg_norm_max_scores_Tesla", "avg_norm_min_scores_Tesla",
    "avg_norm_avg_scores_Apple", "avg_norm_max_scores_Apple", "avg_norm_min_scores_Apple",
    "avg_norm_avg_scores_Amazon", "avg_norm_max_scores_Amazon", "avg_norm_min_scores_Amazon"
]].head())
'''

'\nimport numpy as np\nimport itertools\n\ndef normalize_list(lst):\n    lst = np.array(lst, dtype=float)\n    if len(lst) == 0:\n        return lst\n    max_val = lst.max()\n    min_val = lst.min()\n    def norm(x):\n        if x > 0:\n            return x / max_val if max_val != 0 else 0\n        elif x < 0:\n            return x / abs(min_val) if min_val != 0 else 0\n        else:\n            return 0\n    return np.array([norm(x) for x in lst])\n\nfor stock, col in zip(["Tesla", "Apple", "Amazon"], ["scores_Tesla", "scores_Apple", "scores_Amazon"]):\n    def process(row, col=col):\n        lst = row[col]\n        if not isinstance(lst, list) or len(lst) == 0:\n            return pd.Series({\n                f"avg_norm_avg_{col}": np.nan,\n                f"avg_norm_max_{col}": np.nan,\n                f"avg_norm_min_{col}": np.nan,\n            })\n\n        # For each inner list, compute avg, max, min\n        avgs, maxs, mins = [], [], []\n        for inner in lst:\n            

In [84]:
#df_clean

,timestamp,AAPL,AMZN,TSLA,day,hour,scores_Tesla,scores_Apple,scores_Amazon,avg_norm_avg_scores_Tesla,avg_norm_max_scores_Tesla,avg_norm_min_scores_Tesla,avg_norm_avg_scores_Apple,avg_norm_max_scores_Apple,avg_norm_min_scores_Apple,avg_norm_avg_scores_Amazon,avg_norm_max_scores_Amazon,avg_norm_min_scores_Amazon
0,2018-01-02 09:00:00+00:00,169.1800,1170.0000,311.80,2018-01-02,9,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-02 10:00:00+00:00,169.3500,1169.7000,311.50,2018-01-02,10,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-01-02 11:00:00+00:00,169.6700,NaN,311.89,2018-01-02,11,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-01-02 12:00:00+00:00,170.0500,1173.8900,311.00,2018-01-02,12,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-01-02 13:00:00+00:00,170.1200,1174.5000,311.50,2018-01-02,13,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23526,2023-12-29 20:00:00+00:00,192.5200,151.9500,248.49,2023-12-29,20,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23527,2023-12-29 21:00:00+00:00,192.3800,151.7600,248.40,2023-12-29,21,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23528,2023-12-29 22:00:00+00:00,192.2600,151.7485,248.41,2023-12-29,22,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23529,2023-12-29 23:00:00+00:00,192.1894,151.7700,248.38,2023-12-29,23,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
#df_clean["avg_norm_avg_scores_Tesla"].unique()

array([       nan, 1.        , 0.88235294, ..., 0.61444444, 0.28535354,
       0.30681177])

In [157]:
import numpy as np
import pandas as pd

def normalize_list(lst):
    lst = np.array(lst, dtype=float)
    if len(lst) == 0:
        return lst
    max_val = lst.max()
    min_val = lst.min()
    def norm(x):
        if x > 0:
            return x / max_val if max_val != 0 else 0
        elif x < 0:
            return x / abs(min_val) if min_val != 0 else 0
        else:
            return 0
    return np.array([norm(x) for x in lst])

def add_norm_features(df, col):
    def process(row):
        lst = row[col]
        if not isinstance(lst, list) or len(lst) == 0:
            return pd.Series({
                f"avg_norm_avg_{col}": np.nan,
                f"avg_norm_max_{col}": np.nan,
                f"avg_norm_min_{col}": np.nan,
            })
        avgs, maxs, mins = [], [], []
        for inner in lst:
            if isinstance(inner, list) and len(inner) > 0:
                arr = np.array(inner, dtype=float)
                avgs.append(np.mean(arr))
                maxs.append(np.max(arr))
                mins.append(np.min(arr))
        if len(avgs) == 0:
            return pd.Series({
                f"avg_norm_avg_{col}": np.nan,
                f"avg_norm_max_{col}": np.nan,
                f"avg_norm_min_{col}": np.nan,
            })
        return pd.Series({
            f"avg_norm_avg_{col}": np.mean(normalize_list(avgs)),
            f"avg_norm_max_{col}": np.mean(normalize_list(maxs)),
            f"avg_norm_min_{col}": np.mean(normalize_list(mins)),
        })

    df[[f"avg_norm_avg_{col}", f"avg_norm_max_{col}", f"avg_norm_min_{col}"]] = df.apply(process, axis=1)
    return df

df_apple_clean  = add_norm_features(df_apple_clean,  "scores_Apple")
df_amazon_clean = add_norm_features(df_amazon_clean, "scores_Amazon")
df_tesla_clean  = add_norm_features(df_tesla_clean,  "scores_Tesla")

print(df_apple_clean.head())
print(df_amazon_clean.head())
print(df_tesla_clean.head())

                  timestamp    AAPL         day  hour scores_Apple  \
0 2018-01-02 09:00:00+00:00  169.18  2018-01-02     9           []   
1 2018-01-02 10:00:00+00:00  169.35  2018-01-02    10           []   
2 2018-01-02 11:00:00+00:00  169.67  2018-01-02    11           []   
3 2018-01-02 12:00:00+00:00  170.05  2018-01-02    12           []   
4 2018-01-02 13:00:00+00:00  170.12  2018-01-02    13           []   

   avg_norm_avg_scores_Apple  avg_norm_max_scores_Apple  \
0                        NaN                        NaN   
1                        NaN                        NaN   
2                        NaN                        NaN   
3                        NaN                        NaN   
4                        NaN                        NaN   

   avg_norm_min_scores_Apple  
0                        NaN  
1                        NaN  
2                        NaN  
3                        NaN  
4                        NaN  
                  timestamp     AMZN  

In [161]:
'''
cols = [
    "avg_norm_avg_scores_Tesla", "avg_norm_max_scores_Tesla", "avg_norm_min_scores_Tesla",
    "avg_norm_avg_scores_Apple", "avg_norm_max_scores_Apple", "avg_norm_min_scores_Apple",
    "avg_norm_avg_scores_Amazon", "avg_norm_max_scores_Amazon", "avg_norm_min_scores_Amazon"
]

df_clean[cols] = df_clean[cols].fillna(0)
'''
for df_x, stock in [(df_apple_clean, "Apple"), (df_amazon_clean, "Amazon"), (df_tesla_clean, "Tesla")]:
    cols = [f"avg_norm_avg_scores_{stock}", f"avg_norm_max_scores_{stock}", f"avg_norm_min_scores_{stock}"]
    df_x[cols] = df_x[cols].fillna(0)

In [88]:
#df_clean

,timestamp,AAPL,AMZN,TSLA,day,hour,scores_Tesla,scores_Apple,scores_Amazon,avg_norm_avg_scores_Tesla,avg_norm_max_scores_Tesla,avg_norm_min_scores_Tesla,avg_norm_avg_scores_Apple,avg_norm_max_scores_Apple,avg_norm_min_scores_Apple,avg_norm_avg_scores_Amazon,avg_norm_max_scores_Amazon,avg_norm_min_scores_Amazon
0,2018-01-02 09:00:00+00:00,169.1800,1170.0000,311.80,2018-01-02,9,[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2018-01-02 10:00:00+00:00,169.3500,1169.7000,311.50,2018-01-02,10,[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018-01-02 11:00:00+00:00,169.6700,NaN,311.89,2018-01-02,11,[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2018-01-02 12:00:00+00:00,170.0500,1173.8900,311.00,2018-01-02,12,[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018-01-02 13:00:00+00:00,170.1200,1174.5000,311.50,2018-01-02,13,[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23526,2023-12-29 20:00:00+00:00,192.5200,151.9500,248.49,2023-12-29,20,[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23527,2023-12-29 21:00:00+00:00,192.3800,151.7600,248.40,2023-12-29,21,[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23528,2023-12-29 22:00:00+00:00,192.2600,151.7485,248.41,2023-12-29,22,[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23529,2023-12-29 23:00:00+00:00,192.1894,151.7700,248.38,2023-12-29,23,[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [162]:
df_apple_clean

,timestamp,AAPL,day,hour,scores_Apple,avg_norm_avg_scores_Apple,avg_norm_max_scores_Apple,avg_norm_min_scores_Apple
0,2018-01-02 09:00:00+00:00,169.1800,2018-01-02,9,[],0.0,0.0,0.0
1,2018-01-02 10:00:00+00:00,169.3500,2018-01-02,10,[],0.0,0.0,0.0
2,2018-01-02 11:00:00+00:00,169.6700,2018-01-02,11,[],0.0,0.0,0.0
3,2018-01-02 12:00:00+00:00,170.0500,2018-01-02,12,[],0.0,0.0,0.0
4,2018-01-02 13:00:00+00:00,170.1200,2018-01-02,13,[],0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
23495,2023-12-29 20:00:00+00:00,192.5200,2023-12-29,20,[],0.0,0.0,0.0
23496,2023-12-29 21:00:00+00:00,192.3800,2023-12-29,21,[],0.0,0.0,0.0
23497,2023-12-29 22:00:00+00:00,192.2600,2023-12-29,22,[],0.0,0.0,0.0
23498,2023-12-29 23:00:00+00:00,192.1894,2023-12-29,23,[],0.0,0.0,0.0


In [163]:
df_tesla_clean

,timestamp,TSLA,day,hour,scores_Tesla,avg_norm_avg_scores_Tesla,avg_norm_max_scores_Tesla,avg_norm_min_scores_Tesla
0,2018-01-02 09:00:00+00:00,311.80,2018-01-02,9,[],0.0,0.0,0.0
1,2018-01-02 10:00:00+00:00,311.50,2018-01-02,10,[],0.0,0.0,0.0
2,2018-01-02 11:00:00+00:00,311.89,2018-01-02,11,[],0.0,0.0,0.0
3,2018-01-02 12:00:00+00:00,311.00,2018-01-02,12,[],0.0,0.0,0.0
4,2018-01-02 13:00:00+00:00,311.50,2018-01-02,13,[],0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
23407,2023-12-29 20:00:00+00:00,248.49,2023-12-29,20,[],0.0,0.0,0.0
23408,2023-12-29 21:00:00+00:00,248.40,2023-12-29,21,[],0.0,0.0,0.0
23409,2023-12-29 22:00:00+00:00,248.41,2023-12-29,22,[],0.0,0.0,0.0
23410,2023-12-29 23:00:00+00:00,248.38,2023-12-29,23,[],0.0,0.0,0.0


In [164]:
df_amazon_clean

,timestamp,AMZN,day,hour,scores_Amazon,avg_norm_avg_scores_Amazon,avg_norm_max_scores_Amazon,avg_norm_min_scores_Amazon
0,2018-01-02 09:00:00+00:00,1170.0000,2018-01-02,9,[],0.0,0.0,0.0
1,2018-01-02 10:00:00+00:00,1169.7000,2018-01-02,10,[],0.0,0.0,0.0
2,2018-01-02 12:00:00+00:00,1173.8900,2018-01-02,12,[],0.0,0.0,0.0
3,2018-01-02 13:00:00+00:00,1174.5000,2018-01-02,13,[],0.0,0.0,0.0
4,2018-01-02 14:00:00+00:00,1181.4300,2018-01-02,14,[],0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
22378,2023-12-29 20:00:00+00:00,151.9500,2023-12-29,20,[],0.0,0.0,0.0
22379,2023-12-29 21:00:00+00:00,151.7600,2023-12-29,21,[],0.0,0.0,0.0
22380,2023-12-29 22:00:00+00:00,151.7485,2023-12-29,22,[],0.0,0.0,0.0
22381,2023-12-29 23:00:00+00:00,151.7700,2023-12-29,23,[],0.0,0.0,0.0


In [89]:
#df_clean = df_clean.drop(columns=["day","hour","scores_Tesla", "scores_Apple","scores_Amazon"])

In [94]:
#df_clean

,timestamp,AAPL,AMZN,TSLA,avg_norm_avg_scores_Tesla,avg_norm_max_scores_Tesla,avg_norm_min_scores_Tesla,avg_norm_avg_scores_Apple,avg_norm_max_scores_Apple,avg_norm_min_scores_Apple,avg_norm_avg_scores_Amazon,avg_norm_max_scores_Amazon,avg_norm_min_scores_Amazon
0,2018-01-02 09:00:00+00:00,169.1800,1170.0000,311.80,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2018-01-02 10:00:00+00:00,169.3500,1169.7000,311.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018-01-02 11:00:00+00:00,169.6700,NaN,311.89,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2018-01-02 12:00:00+00:00,170.0500,1173.8900,311.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018-01-02 13:00:00+00:00,170.1200,1174.5000,311.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
23526,2023-12-29 20:00:00+00:00,192.5200,151.9500,248.49,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23527,2023-12-29 21:00:00+00:00,192.3800,151.7600,248.40,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23528,2023-12-29 22:00:00+00:00,192.2600,151.7485,248.41,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23529,2023-12-29 23:00:00+00:00,192.1894,151.7700,248.38,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [165]:
df_apple_clean  = df_apple_clean.drop(columns=["scores_Apple"])
df_amazon_clean = df_amazon_clean.drop(columns=[ "scores_Amazon"])
df_tesla_clean  = df_tesla_clean.drop(columns=[ "scores_Tesla"])

In [166]:
df_apple_clean["AAPL"].min()

104.05

In [169]:
df_apple_clean

,timestamp,AAPL,day,hour,avg_norm_avg_scores_Apple,avg_norm_max_scores_Apple,avg_norm_min_scores_Apple
0,2018-01-02 09:00:00+00:00,169.1800,2018-01-02,9,0.0,0.0,0.0
1,2018-01-02 10:00:00+00:00,169.3500,2018-01-02,10,0.0,0.0,0.0
2,2018-01-02 11:00:00+00:00,169.6700,2018-01-02,11,0.0,0.0,0.0
3,2018-01-02 12:00:00+00:00,170.0500,2018-01-02,12,0.0,0.0,0.0
4,2018-01-02 13:00:00+00:00,170.1200,2018-01-02,13,0.0,0.0,0.0
...,...,...,...,...,...,...,...
23495,2023-12-29 20:00:00+00:00,192.5200,2023-12-29,20,0.0,0.0,0.0
23496,2023-12-29 21:00:00+00:00,192.3800,2023-12-29,21,0.0,0.0,0.0
23497,2023-12-29 22:00:00+00:00,192.2600,2023-12-29,22,0.0,0.0,0.0
23498,2023-12-29 23:00:00+00:00,192.1894,2023-12-29,23,0.0,0.0,0.0


In [167]:
df_amazon_clean["AMZN"].min()

81.83

In [168]:
df_tesla_clean["TSLA"].min()

103.1

In [171]:
df_apple_clean = df_apple_clean.rename(columns={
    "AAPL": "close",
    "avg_norm_avg_scores_Apple":"avg_norm_avg_scores",
    "avg_norm_max_scores_Apple": "avg_norm_max_scores",
    "avg_norm_min_scores_Apple":"avg_norm_min_scores"
})

In [172]:
df_tesla_clean = df_tesla_clean.rename(columns={
    "TSLA": "close",
    "avg_norm_avg_scores_Tesla":"avg_norm_avg_scores",
    "avg_norm_max_scores_Tesla": "avg_norm_max_scores",
    "avg_norm_min_scores_Tesla":"avg_norm_min_scores"
})

In [173]:
df_amazon_clean = df_amazon_clean.rename(columns={
    "AMZN": "close",
    "avg_norm_avg_scores_Amazon":"avg_norm_avg_scores",
    "avg_norm_max_scores_Amazon": "avg_norm_max_scores",
    "avg_norm_min_scores_Amazon":"avg_norm_min_scores"
})

In [174]:
df_tesla_clean.to_csv("Tesla_processed_news_scores_with_stocks_WITH_AGGREGATE_ready_formodeling.csv", index=False)
df_apple_clean.to_csv("Apple_processed_news_scores_with_stocks_WITH_AGGREGATE_ready_formodeling.csv", index=False)
df_amazon_clean.to_csv("Amazon_processed_news_scores_with_stocks_WITH_AGGREGATE_ready_formodeling.csv", index=False)

In [93]:

#df_clean.to_csv("processed_news_scores_with_stocks_WITH_AGGREGATE_ready_formodeling.csv", index=False)